# AMPS II · Week 10 — Lab
## Cohesive interfaces: traction–separation laws, submeshes, snap-back and path following

| | |
|---|---|
| **Lecture sections** | Part A (cohesive laws, the bar) · Part B (submeshes, the '+' side) · Part C (path following, the plate) · Part D (springs everywhere, LLM) |
| **Time** | ≈ 80 min in class |
| **You will** | check a cohesive law in NumPy · reproduce an imperfect interface exactly and see the '+' trap · trace a snap-back exactly with opening control and watch displacement control jump · debond the source's inclusions and check both ends of the curve · judge an LLM's interface script |

**Reminder.** `%%fenicsx` cells are separate MPI programs; ordinary cells have NumPy and matplotlib only. The model module `czm.py` is written once with `%%writefile` (§1). All cells run with `-np 2`.

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## 1 · The model module  *(lecture §A1–D1)*
`czm.py`: meshes (`strip_mesh`, `plate_mesh` — Gmsh with `shared_facet` ghosts), `interface_data` (the interface facets with subdomain 1 as '+'), `Bimaterial` (one space per submesh, laws
`"linear"`, `"exp"`, `"bilinear"`; `.fix`, `.setup(load_side, control)`, `.solve_load`, `.solve_opening`, `.force`, `.end_disp`, `.opening`, `.interface_jumps`), references (`T_exp`, `strip_exact`,
`bonded_plate`) and the source's model with springs on every facet (`dg_intrinsic`). Run the cell.

In [ ]:
%%writefile /tmp/fnx/czm.py
"""Cohesive interfaces, DOLFINx 0.11. Two subdomains (cell tags 1, 2), each with its own Lagrange space on a submesh, joined on the
interface Gamma by a traction-separation law T([[u]]), [[u]] = u2 - u1:
    sum_i int_{Omega_i} sigma(u_i) : eps(v_i) dx + int_Gamma T([[u]]) . [[v]] dS = lam int_{right edge} e_x . v ds
Laws (isotropic in the opening norm d = |[[u]]|): linear T = K [[u]];  exponential (the source) T = Gc/d0^2 exp(-d/d0) [[u]], d0 = Gc/(e sc);
bilinear: T = K0 [[u]] up to d = sc/K0, then the secant decreases to 0 at dc = 2 Gc / sc.
ORDERING: the interior-facet data are built explicitly with the cell of subdomain 1 as '+'. With plain meshtags DOLFINx orders the two cells
of a facet as it likes (it depends on the mesh and on the partition): u1('+') is then sometimes evaluated in a cell of subdomain 2."""
import gc
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import NonlinearProblem, LinearProblem
import dolfinx.fem.petsc as fp

comm = MPI.COMM_WORLD
# MPI + Python garbage collection: a problem object that is replaced (b = Bimaterial(...) in a loop) is freed by the cyclic garbage
# collector at a DIFFERENT moment on each rank; its __del__ destroys PETSc objects collectively -> one rank waits in the destroy, the
# other in the next assembly: a deadlock (seen here in about half of the -np 2 runs). Automatic collection is switched off and every
# constructor collects explicitly, i.e. at the same point on all ranks.
gc.disable()
def collect():
    gc.collect()
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

LU = {"ksp_type": "preonly", "pc_type": "lu", "pc_factor_mat_solver_type": "mumps"}

# ---------------------------------------------------------------- meshes and tags
def tag_cells(msh, f):
    """cell tags from midpoints: f(x) -> 1 or 2"""
    tdim = msh.topology.dim; nc = msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts
    mid = mesh.compute_midpoints(msh, tdim, np.arange(nc, dtype=np.int32))
    return mesh.meshtags(msh, tdim, np.arange(nc, dtype=np.int32), f(mid.T).astype(np.int32))

def strip_mesh(nx=8, ny=4, L=2.0, H=1.0):
    """[0, L] x [0, H] with quadrilaterals; subdomain 1: x < L/2, subdomain 2: x > L/2 (interface x = L/2, a mesh line if nx is even)"""
    msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], mesh.CellType.quadrilateral)
    return msh, tag_cells(msh, lambda x: np.where(x[0] < L / 2, 1, 2))

def plate_mesh(h, Lx=2.0, Ly=1.0, R=0.2, centres=((0.5, 0.5), (1.5, 0.5)), holes=False):
    """the source's plate with two circular inclusions (Gmsh fragment: the interface nodes are shared); tags 1 matrix, 2 inclusions.
    holes=True: the inclusions are removed (the fully debonded limit)"""
    import gmsh
    from dolfinx.io import gmsh as dg
    gmsh.initialize(); gmsh.option.setNumber("General.Verbosity", 0); gmsh.model.add("plate")
    if comm.rank == 0:
        occ = gmsh.model.occ
        rect = occ.addRectangle(0, 0, 0, Lx, Ly); discs = [(2, occ.addDisk(cx, cy, 0, R, R)) for cx, cy in centres]
        (occ.cut if holes else occ.fragment)([(2, rect)], discs); occ.synchronize()
        surfs = [t for _, t in gmsh.model.getEntities(2)]
        mat = [t for t in surfs if gmsh.model.isInside(2, t, [0.0, 0.0, 0.0])]
        gmsh.model.addPhysicalGroup(2, mat, 1)
        if not holes:
            gmsh.model.addPhysicalGroup(2, [t for t in surfs if t not in mat], 2)
        gmsh.option.setNumber("Mesh.CharacteristicLengthMax", h); gmsh.option.setNumber("Mesh.CharacteristicLengthMin", h)
        gmsh.model.mesh.generate(2)
    # interior-facet integrals (the interface, DG coupling) need the cells on BOTH sides of a facet on the rank that owns it:
    # Gmsh meshes are created without ghost cells by default -> facets on a partition boundary would be silently skipped
    md = dg.model_to_mesh(gmsh.model, comm, 0, gdim=2, partitioner=mesh.create_cell_partitioner(mesh.GhostMode.shared_facet, 2)); gmsh.finalize()
    return md.mesh, md.cell_tags

def interface_data(msh, ct, a=1, b=2):
    """owned interior facets between cells tagged a and b as integration data [cell_a, local facet, cell_b, local facet, ...]: '+' = side a.
    Also returns the facet indices."""
    tdim = msh.topology.dim; fdim = tdim - 1
    msh.topology.create_connectivity(fdim, tdim); msh.topology.create_connectivity(tdim, fdim)
    f2c, c2f = msh.topology.connectivity(fdim, tdim), msh.topology.connectivity(tdim, fdim)
    tag = np.zeros(msh.topology.index_map(tdim).size_local + msh.topology.index_map(tdim).num_ghosts, dtype=np.int32)
    tag[ct.indices] = ct.values
    data, facets = [], []
    for f in range(msh.topology.index_map(fdim).size_local):
        cs = f2c.links(f)
        if len(cs) == 2 and {tag[cs[0]], tag[cs[1]]} == {a, b}:
            c0, c1 = (cs[0], cs[1]) if tag[cs[0]] == a else (cs[1], cs[0])
            data += [c0, int(np.where(c2f.links(c0) == f)[0][0]), c1, int(np.where(c2f.links(c1) == f)[0][0])]
            facets.append(f)
    return np.array(data, dtype=np.int32), np.array(facets, dtype=np.int32)

# ---------------------------------------------------------------- 1D references
def T_exp(d, Gc, sc):
    """exponential law (normal opening d >= 0): T = Gc/d0^2 d exp(-d/d0), peak sc at d = d0 = Gc/(e sc), integral Gc"""
    d0 = Gc / (np.e * sc)
    return Gc / d0**2 * d * np.exp(-d / d0)

def strip_exact(d, L, E, Gc, sc):
    """bar/strip of length L (uniaxial stress) with one interface: sigma = T(d), end displacement u = sigma L / E + d"""
    s = T_exp(d, Gc, sc)
    return s, s * L / E + d

# ---------------------------------------------------------------- the interface model
class Bimaterial:
    count = 0
    def __init__(self, msh, ct, E=(10.0, 70.0), nu=(0.3, 0.2), law="exp", K=None, Gc=0.01, sc=0.5, deg=1, plane="stress", order="explicit"):
        """E, nu: (subdomain 1, subdomain 2); law "linear" (K), "exp" (Gc, sc), "bilinear" (K, Gc, sc); order "explicit" or "meshtags".
        The laws are isotropic in |[[u]]| (as in the source): they soften in compression too — no contact, interpenetration is possible."""
        collect()
        self.msh, self.ct = msh, ct; tdim = msh.topology.dim; fdim = tdim - 1
        subs = [mesh.create_submesh(msh, tdim, ct.find(i)) for i in (1, 2)]
        self.sub, self.em = [s[0] for s in subs], [s[1] for s in subs]
        self.V = [fem.functionspace(s, ("Lagrange", deg, (2,))) for s in self.sub]
        self.u = [fem.Function(V) for V in self.V]; self.w = [ufl.TestFunction(V) for V in self.V]
        data, self.ifacets = interface_data(msh, ct)
        if order == "explicit":
            self.dS = ufl.Measure("dS", domain=msh, subdomain_data=[(1, data)])(1)
        else:
            fs = np.sort(self.ifacets)
            self.dS = ufl.Measure("dS", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, fs, np.ones(len(fs), dtype=np.int32)))(1)
        self.dx = ufl.Measure("dx", domain=msh, subdomain_data=ct)
        def lame(Ei, ni):
            lam, mu = Ei * ni / ((1 + ni) * (1 - 2 * ni)), Ei / (2 * (1 + ni))
            return (2 * lam * mu / (lam + 2 * mu) if plane == "stress" else lam), mu
        self.mat = [lame(E[i], nu[i]) for i in (0, 1)]
        self.sig = lambda u, i: self.mat[i][0] * ufl.tr(ufl.sym(ufl.grad(u))) * ufl.Identity(2) + 2 * self.mat[i][1] * ufl.sym(ufl.grad(u))
        self.jmp = self.u[1]("-") - self.u[0]("+")
        self.T = self.law(self.jmp, law, K, Gc, sc)
        n1 = ufl.FacetNormal(msh)("+")                                 # outward normal of side 1: the normal gap is [[u]] . n1
        self.gap = ufl.dot(self.jmp, n1)
        self.lam = fem.Constant(msh, 0.0)
        self.Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
        right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], self.Lx))
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, np.sort(right), np.ones(len(right), dtype=np.int32)))(1)
        self.F = [ufl.inner(self.sig(self.u[0], 0), ufl.sym(ufl.grad(self.w[0]))) * self.dx(1) - ufl.dot(self.T, self.w[0]("+")) * self.dS,
                  ufl.inner(self.sig(self.u[1], 1), ufl.sym(ufl.grad(self.w[1]))) * self.dx(2) + ufl.dot(self.T, self.w[1]("-")) * self.dS]
        self.bcs = []; self.n = ufl.FacetNormal(msh)
        self.glen = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * self.dS)), op=MPI.SUM)

    def law(self, j, law, K, Gc, sc):
        d = ufl.sqrt(ufl.dot(j, j) + 1e-30)
        if law == "linear":
            return K * j
        if law == "exp":
            self.d0 = d0 = Gc / (np.e * sc)
            return Gc / d0**2 * ufl.exp(-d / d0) * j
        if law == "bilinear":                                         # secant K0 up to d1 = sc/K0, then sc (dc - d)/(dc - d1)/d, 0 beyond dc
            d1, dc = sc / K, 2 * Gc / sc; self.d0 = d1
            sec = ufl.conditional(ufl.lt(d, d1), K, ufl.conditional(ufl.lt(d, dc), sc * (dc - d) / ((dc - d1) * d), 0.0))
            return sec * j
        raise ValueError(law)

    def fix(self, i, marker, comp):
        """u_comp = 0 on the boundary facets of subdomain i (0 or 1) selected by marker"""
        V = self.V[i]; Vc, _ = V.sub(comp).collapse(); fdim = self.sub[i].topology.dim - 1
        f = mesh.locate_entities_boundary(self.sub[i], fdim, marker)
        self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_topological((V.sub(comp), Vc), fdim, f), V.sub(comp)))

    def fix_point(self, i, marker, comp):
        V = self.V[i]; Vc, _ = V.sub(comp).collapse()
        self.bcs.append(fem.dirichletbc(fem.Function(Vc), fem.locate_dofs_geometrical((V.sub(comp), Vc), marker), V.sub(comp)))

    def setup(self, load_side=0, control="traction"):
        """the right edge belongs to subdomain load_side (0 or 1); control "traction" (lam = traction) or "displacement" (lam = u_x there)"""
        i = self.load_side = load_side; self.control = control
        F = list(self.F)
        if control == "traction":
            F[i] = F[i] - self.lam * self.w[i][0] * self.ds
        else:
            V = self.V[i]; Vc, _ = V.sub(0).collapse(); fdim = self.sub[i].topology.dim - 1
            self.ubc = fem.Function(Vc)
            f = mesh.locate_entities_boundary(self.sub[i], fdim, lambda x: np.isclose(x[0], self.Lx))
            self.bcs.append(fem.dirichletbc(self.ubc, fem.locate_dofs_topological((V.sub(0), Vc), fdim, f), V.sub(0)))
        Bimaterial.count += 1
        self.pb = NonlinearProblem(F, self.u, bcs=self.bcs, petsc_options_prefix=f"bm{Bimaterial.count}_", entity_maps=self.em, kind="mpi",
                                   petsc_options={**LU, "snes_type": "newtonls", "snes_linesearch_type": "none", "snes_max_it": 30,
                                                  "snes_rtol": 1e-10, "snes_atol": 1e-12})
        n = self.n          # opening functional g(u) = mean normal opening (1/|Gamma|) int (u2 - u1) . n1 dS, n1 = n('+') (outward normal of side 1)
        self.Gm = [fem.form(-ufl.dot(self.w[0]("+"), n("+")) * self.dS, entity_maps=self.em),
                   fem.form(ufl.dot(self.w[1]("-"), n("+")) * self.dS, entity_maps=self.em)]
        self.f_open = fem.form(ufl.dot(self.jmp, n("+")) * self.dS, entity_maps=self.em)
        self.f_ux = fem.form(self.u[i][0] * self.ds, entity_maps=self.em); self.f_len = fem.form(1.0 * self.ds)
        self.f_reac = fem.form(ufl.dot(ufl.dot(self.sig(self.u[i], i), n), ufl.as_vector([1.0, 0.0])) * self.ds, entity_maps=self.em)

    def solve_load(self, value):
        """load (traction) or displacement control: Newton (SNES) at a fixed lam. Returns (SNES reason, iterations)."""
        if self.control == "traction":
            self.lam.value = value
        else:
            self.ubc.x.array[:] = value
        self.pb.solve()
        return self.pb.solver.getConvergedReason(), self.pb.solver.getIterationNumber()

    def solve_opening(self, target, tol=1e-9, maxit=30):
        """opening control (traction loading): solve R(u) - lam f = 0 together with g(u) = target, a bordered Newton step with two
        right-hand sides per iteration (K a = -R, K b = f, dlam from the constraint). Returns (iterations, |R|)."""
        pb = self.pb; x = pb.x; A = pb.A; ksp = pb.solver.getKSP()
        fp.assign(self.u, x)
        g = fp.assemble_vector(self.Gm, kind="mpi"); g.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); g.scale(1.0 / self.glen)
        R, R1 = pb.b.duplicate(), pb.b.duplicate(); r0 = None
        for it in range(maxit):
            lam0 = float(self.lam.value)
            pb.solver.computeFunction(x, R)
            self.lam.value = lam0 + 1.0; pb.solver.computeFunction(x, R1); self.lam.value = lam0
            f = R.copy(); f.axpy(-1.0, R1)                             # f = dR/d(-lam): the load vector
            nr = R.norm(); r0 = r0 or max(nr, 1e-30); res = target - g.dot(x)
            if it > 0 and nr < tol * max(r0, 1.0) and abs(res) < tol * max(abs(target), 1e-12):
                break
            pb.solver.computeJacobian(x, A, A); ksp.setOperators(A)
            a, b = R.duplicate(), R.duplicate(); mR = R.copy(); mR.scale(-1.0)
            ksp.solve(mR, a); ksp.solve(f, b)
            dl = (res - g.dot(a)) / g.dot(b)
            x.axpy(1.0, a); x.axpy(dl, b); self.lam.value = lam0 + dl
        fp.assign(x, self.u)
        for u in self.u:
            u.x.scatter_forward()
        return it, nr

    def opening(self):
        """mean normal opening of the interface"""
        return comm.allreduce(fem.assemble_scalar(self.f_open), op=MPI.SUM) / self.glen

    def end_disp(self):
        """mean u_x on the loaded (right) edge"""
        return comm.allreduce(fem.assemble_scalar(self.f_ux), op=MPI.SUM) / comm.allreduce(fem.assemble_scalar(self.f_len), op=MPI.SUM)

    def force(self):
        """force on the right edge: lam * height (traction control) or the integral of sigma_xx (displacement control)"""
        if self.control == "traction":
            return float(self.lam.value) * comm.allreduce(fem.assemble_scalar(self.f_len), op=MPI.SUM)
        return comm.allreduce(fem.assemble_scalar(self.f_reac), op=MPI.SUM)

    def interface_jumps(self):
        """(facet midpoints, [[u]] there) of all interface facets, gathered on rank 0 (point evaluation in each submesh)"""
        from dolfinx import geometry
        mid = mesh.compute_midpoints(self.msh, self.msh.topology.dim - 1, self.ifacets)
        allmid = comm.allgather(mid); P = np.vstack(allmid) if sum(len(m) for m in allmid) else np.zeros((0, 3))
        vals = []
        for k in (0, 1):
            m = self.sub[k]; tdim = m.topology.dim; nloc = m.topology.index_map(tdim).size_local
            cand = geometry.compute_colliding_cells(m, geometry.compute_collisions_points(geometry.bb_tree(m, tdim), P), P)
            out = np.zeros((len(P), 2)); cnt = np.zeros(len(P))
            for j in range(len(P)):
                own = [c for c in cand.links(j) if c < nloc]
                if own:
                    out[j] = np.asarray(self.u[k].eval(P[j], [own[0]])).reshape(-1); cnt[j] = 1
            out = comm.allreduce(out, op=MPI.SUM); cnt = comm.allreduce(cnt, op=MPI.SUM)
            vals.append(out / np.maximum(cnt, 1)[:, None])
        return P, vals[1] - vals[0]

# ---------------------------------------------------------------- references
def bonded_plate(h, E=(10.0, 70.0), nu=(0.3, 0.2), holes=False, load=0.1, deg=2):
    """single CG mesh (perfect bond) or the plate with holes; left u_x = 0, bottom u_y = 0, traction load * e_x on the right (plane stress).
    Returns force / mean end displacement."""
    collect()
    msh, ct = plate_mesh(h, holes=holes)
    Q = fem.functionspace(msh, ("DG", 0)); Ef, nf = fem.Function(Q), fem.Function(Q)
    Ef.x.array[:] = E[0]; nf.x.array[:] = nu[0]
    if not holes:
        Ef.x.array[ct.find(2)] = E[1]; nf.x.array[ct.find(2)] = nu[1]
    lam, mu = Ef * nf / (1 - nf**2), Ef / 2 / (1 + nf)
    V = fem.functionspace(msh, ("Lagrange", deg, (2,))); u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    sig = lambda w: lam * ufl.tr(ufl.sym(ufl.grad(w))) * ufl.Identity(2) + 2 * mu * ufl.sym(ufl.grad(w))
    fdim = 1; V0, _ = V.sub(0).collapse(); V1, _ = V.sub(1).collapse()
    bcs = [fem.dirichletbc(fem.Function(V0), fem.locate_dofs_topological((V.sub(0), V0), fdim, mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0))), V.sub(0)),
           fem.dirichletbc(fem.Function(V1), fem.locate_dofs_topological((V.sub(1), V1), fdim, mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))), V.sub(1))]
    Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
    right = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], Lx))
    ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, np.sort(right), np.ones(len(right), dtype=np.int32)))(1)
    Bimaterial.count += 1
    uh = LinearProblem(ufl.inner(sig(u), ufl.sym(ufl.grad(v))) * ufl.dx, load * v[0] * ds, bcs=bcs, petsc_options=LU,
                       petsc_options_prefix=f"bond{Bimaterial.count}_").solve()
    H = comm.allreduce(fem.assemble_scalar(fem.form(1.0 * ds)), op=MPI.SUM)
    ub = comm.allreduce(fem.assemble_scalar(fem.form(uh[0] * ds)), op=MPI.SUM) / H
    return load * H / ub

def dg_intrinsic(msh, ct, K_int, K_bulk, E=(10.0, 70.0), nu=(0.3, 0.2), beta=1e3, u_app=0.002):
    """the source's model, linearised: DG1 everywhere, a cohesive spring on EVERY interior facet (K_int on the interface, K_bulk elsewhere),
    Nitsche for the Dirichlet conditions (left u_x = u_y = 0, bottom u_y = 0, right u_x = u_app). Returns reaction force / u_app."""
    collect()
    tdim = msh.topology.dim; fdim = tdim - 1
    Q = fem.functionspace(msh, ("DG", 0)); Ef, nf = fem.Function(Q), fem.Function(Q)
    Ef.x.array[:] = E[0]; nf.x.array[:] = nu[0]; Ef.x.array[ct.find(2)] = E[1]; nf.x.array[ct.find(2)] = nu[1]
    lam, mu = Ef * nf / (1 - nf**2), Ef / 2 / (1 + nf)
    V = fem.functionspace(msh, ("DG", 1, (2,))); u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
    sig = lambda w: lam * ufl.tr(ufl.sym(ufl.grad(w))) * ufl.Identity(2) + 2 * mu * ufl.sym(ufl.grad(w))
    data, ifac = interface_data(msh, ct)
    msh.topology.create_connectivity(fdim, tdim)
    allint = np.arange(msh.topology.index_map(fdim).size_local, dtype=np.int32)
    nb = np.array([len(msh.topology.connectivity(fdim, tdim).links(f)) for f in allint]); allint = allint[nb == 2]
    tags = np.where(np.isin(allint, ifac), 1, 2).astype(np.int32)
    dS = ufl.Measure("dS", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, allint, tags))
    Lx = comm.allreduce(msh.geometry.x[:, 0].max(), op=MPI.MAX)
    fl = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], 0)); fr = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[0], Lx))
    fb = mesh.locate_entities_boundary(msh, fdim, lambda x: np.isclose(x[1], 0))
    idx = np.concatenate([fl, fr, fb]); val = np.concatenate([np.full(len(fl), 1), np.full(len(fr), 2), np.full(len(fb), 3)]).astype(np.int32)
    o = np.argsort(idx); ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, fdim, idx[o], val[o]))
    n = ufl.FacetNormal(msh); h = ufl.CellDiameter(msh); ex, ey = ufl.as_vector([1.0, 0.0]), ufl.as_vector([0.0, 1.0])
    def nit(c, g, tag):                                               # Nitsche for the component c (unit vector) = g on ds(tag)
        return (-ufl.dot(ufl.dot(sig(u), n), c) * ufl.dot(v, c) - ufl.dot(ufl.dot(sig(v), n), c) * ufl.dot(u, c)
                + beta / h * ufl.dot(u, c) * ufl.dot(v, c)) * ds(tag), (-ufl.dot(ufl.dot(sig(v), n), c) * g + beta / h * g * ufl.dot(v, c)) * ds(tag)
    a = ufl.inner(sig(u), ufl.sym(ufl.grad(v))) * ufl.dx + K_int * ufl.dot(ufl.jump(u), ufl.jump(v)) * dS(1) + K_bulk * ufl.dot(ufl.jump(u), ufl.jump(v)) * dS(2)
    L = ufl.dot(fem.Constant(msh, np.zeros(2)), v) * ufl.dx
    for c, g, tag in ((ex, 0.0, 1), (ey, 0.0, 1), (ex, u_app, 2), (ey, 0.0, 3)):
        aa, ll = nit(c, g, tag); a += aa; L += ll
    Bimaterial.count += 1
    uh = LinearProblem(a, L, petsc_options=LU, petsc_options_prefix=f"dg{Bimaterial.count}_").solve()
    F = comm.allreduce(fem.assemble_scalar(fem.form(ufl.dot(ufl.dot(sig(uh), n), ex) * ds(2))), op=MPI.SUM)
    return F / u_app

## 2 · The exponential law in NumPy  *(lecture §A1–A2)*
$T(\delta)=\frac{G_c}{\delta_0^2}\delta e^{-\delta/\delta_0}$ with $\delta_0=G_c/(e\sigma_c)$. Check the peak, the fracture energy and the critical length $L_c=E\delta_0^2e^2/G_c$ of the bar ($E=10$, $G_c=0.01$, $\sigma_c=0.5$).

In [ ]:
import numpy as np
Gc, sc, E = 0.01, 0.5, 10.0; d0 = Gc / (np.e * sc)
T = lambda d: Gc / d0**2 * d * np.exp(-d / d0)
d = np.linspace(0, 60 * d0, 600001)
print(f"d0 = {d0:.6f}; peak {T(d).max():.8f} at d = {d[np.argmax(T(d))] / d0:.5f} d0;  integral of T = {np.trapezoid(T(d), d):.8f} (Gc = {Gc})")
dT = np.gradient(T(d), d); print(f"min T' = {dT.min():.4f} at d = {d[np.argmin(dT)] / d0:.4f} d0 (exact -Gc e^-2/d0^2 = {-Gc * np.exp(-2) / d0**2:.4f});  L_c = E/|min T'| = {E / -dT.min():.4f}")

> **✏️ Your turn (3 min).** Why is $L_c$ a property of the bar and not of the law? What happens to $L_c$ if $G_c$ doubles at fixed $\sigma_c$?

> *your answer*

## 3 · An imperfect interface, exactly — and the '+' trap  *(lecture §B1–B2)*
A strip $[0,2]\times[0,1]$ ($E=1$, $\nu=0.3$, plane stress), a linear spring $K=5$ on the interface $x=1$, traction $\sigma=0.01$ on the right: $u_{\rm end}=\sigma L/E+\sigma/K=0.022$ exactly.
Once with the explicit facet data ('+' = left), once with plain meshtags.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from czm import *
Gc, sc, E = 0.01, 0.5, 10.0; d0 = Gc / (np.e * sc)
def strip(law="exp", L=2.0, nx=8, order="explicit", control="traction", E_=(10.0, 10.0), **kw):
    """strip [0, L] x [0, 1], interface x = L/2; left edge u_x = 0 (+ one point u_y = 0); load on the right edge"""
    msh, ct = strip_mesh(nx, 4, L=L)
    b = Bimaterial(msh, ct, E=E_, nu=(0.3, 0.3), law=law, order=order, **kw)
    b.fix(0, lambda x: np.isclose(x[0], 0.0), 0); b.fix_point(0, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0), 1)
    b.setup(load_side=1, control=control)
    return b

for order in ("explicit", "meshtags"):
    b = strip(law="linear", K=5.0, order=order, E_=(1.0, 1.0)); r = b.solve_load(0.01)
    say(f"{order:9s}: u_end = {b.end_disp():.12f} (exact 0.022),  SNES reason {r[0]}, iterations {r[1]}")

> **✏️ Your turn (5 min).** (i) Run the cell once more with `-np 1` (edit the first line). What changes? (ii) Why does the exact solution not depend on the mesh here?

> *your answer*

## 4 · Snap-back: opening control against the exact bar  *(lecture §A2, §C1–C2)*
The strip with the exponential law ($E=10$ on both sides, $L=2>L_c$). Opening control traces the curve; the step grid misses the peak $\delta_0$ on purpose. Then displacement control.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from czm import *
Gc, sc, E = 0.01, 0.5, 10.0; d0 = Gc / (np.e * sc)
def strip(law="exp", L=2.0, nx=8, order="explicit", control="traction", E_=(10.0, 10.0), **kw):
    """strip [0, L] x [0, 1], interface x = L/2; left edge u_x = 0 (+ one point u_y = 0); load on the right edge"""
    msh, ct = strip_mesh(nx, 4, L=L)
    b = Bimaterial(msh, ct, E=E_, nu=(0.3, 0.3), law=law, order=order, **kw)
    b.fix(0, lambda x: np.isclose(x[0], 0.0), 0); b.fix_point(0, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0), 1)
    b.setup(load_side=1, control=control)
    return b

b = strip(Gc=Gc, sc=sc); rows = []; dev = 0.0
for dt in np.arange(0.13, 10.0, 0.13) * d0:
    it, nr = b.solve_opening(dt); s, ue = strip_exact(dt, 2.0, E, Gc, sc)
    rows.append([dt, float(b.lam.value), b.end_disp(), it + 1]); dev = max(dev, abs(rows[-1][1] - s), abs(rows[-1][2] - ue))
R = np.array(rows)
say(f"opening control: max |FE - exact| = {dev:.1e}, Newton iterations <= {int(R[:, 3].max())}; peak {R[:, 1].max():.5f}, u at the peak {R[np.argmax(R[:, 1]), 2]:.5f}, min u after it {R[R[:, 0] > 2 * d0, 2].min():.5f}")
bd = strip(Gc=Gc, sc=sc, control="displacement"); D = []
for ub in np.arange(0.005, 0.2001, 0.005):
    r = bd.solve_load(ub); D.append([ub, bd.force(), r[0], r[1]])
D = np.array(D); k = int(np.argmax(D[:, 1]))
say("displacement control around the peak (u, force, SNES reason, iterations):"); [say("  ", np.round(row, 6)) for row in D[k - 1:k + 3]]
if comm.rank == 0:
    np.savez("/tmp/fnx/strip.npz", R=R, D=D)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load("/tmp/fnx/strip.npz"); R, D = d["R"], d["D"]
Gc, sc, E = 0.01, 0.5, 10.0; d0 = Gc / (np.e * sc); dd = np.linspace(1e-6, 10 * d0, 400); s = Gc / d0**2 * dd * np.exp(-dd / d0)
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(s * 2 / E + dd, s, "k-", lw=1, label="exact"); ax.plot(R[:, 2], R[:, 1], "o", ms=3, mfc="none", label="opening control")
ax.plot(D[:, 0], D[:, 1], "s--", ms=3, label="displacement control")
ax.set(xlabel="end displacement", ylabel="stress", title="strip L = 2"); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** (i) Displacement control reports "converged" right after the peak. Converged to what, and how would you have noticed without the exact curve? (ii) Change the step grid so that a
> step lands exactly on $\delta_0$ (`np.arange(0.1, ...)` with 0.1 steps) and explain what happens.

> *your answer*

## 5 · The plate with two inclusions  *(lecture §C3)*
The source's plate. First the linear interface: $K\to\infty$ must give the perfectly bonded plate (`bonded_plate`, one CG mesh), $K=G_c/\delta_0^2$ the source's initial stiffness. Then the exponential law
with opening control; at the end every interface has lost its stiffness, so the slope must be that of a plate with **holes**.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from czm import *
Gc, sc = 0.01, 0.5; d0 = Gc / (np.e * sc)
def plate(msh, ct, law="exp", control="traction", **kw):
    """the source's plate: matrix (E 10, nu 0.3) with two inclusions (E 70, nu 0.2); left u_x = 0, bottom u_y = 0, load on the right; P2"""
    p = Bimaterial(msh, ct, law=law, Gc=Gc, sc=sc, deg=2, **kw)
    p.fix(0, lambda x: np.isclose(x[0], 0.0), 0); p.fix(0, lambda x: np.isclose(x[1], 0.0), 1)
    p.setup(load_side=0, control=control)
    return p

msh, ct = plate_mesh(0.06)
kb, kh = bonded_plate(0.06), bonded_plate(0.06, holes=True)
say(f"perfect bond {kb:.5f}, holes {kh:.5f}")
for K in (Gc / d0**2, 1e3, 1e6):
    p = plate(msh, ct, law="linear", K=K); p.solve_load(0.1); say(f"linear interface K = {K:9.1f}: stiffness {p.force() / p.end_disp():.5f}")
p = plate(msh, ct); rows = []
for dt in np.concatenate([np.arange(0.13, 1.0, 0.13), np.arange(1.0, 3.5, 0.0625), np.arange(3.5, 8.01, 0.13)]) * d0:
    it, nr = p.solve_opening(dt); rows.append([dt / d0, p.force(), p.end_disp(), it + 1])
R = np.array(rows)
say(f"initial secant {R[0, 1] / R[0, 2]:.4f}; final slope {(R[-1, 1] - R[-8, 1]) / (R[-1, 2] - R[-8, 2]):.5f}; steps at 30 iterations: {int((R[:, 3] >= 30).sum())} of {len(R)}")
if comm.rank == 0:
    np.savez("/tmp/fnx/plate.npz", R=R, kb=kb, kh=kh)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
d = np.load("/tmp/fnx/plate.npz"); R = d["R"]; uu = np.linspace(0, R[:, 2].max(), 50)
fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot(R[:, 2], R[:, 1], "o-", ms=3, label="CZM, opening control"); ax.plot(uu, d["kb"] * uu, "k--", lw=0.8, label="perfect bond"); ax.plot(uu, d["kh"] * uu, "k:", lw=0.8, label="holes")
ax.set(xlabel="mean end displacement", ylabel="force", ylim=(0, 1.1 * R[:, 1].max()), title="debonding"); ax.legend(); ax.grid(alpha=0.3); plt.show()

> **✏️ Your turn (5 min).** (i) Why is the initial stiffness 6 % below perfect bond, and is that a numerical error? (ii) A few steps stop at 30 iterations near the end of debonding. What is singular there?
> (iii) The end slope equals the plate with holes. What does a real debonded inclusion do that this model cannot?

> *your answer*

## 6 · LLM exercise: an imperfect interface between two materials  *(lecture §D2)*
Two bonded layers, $[0,1]\times[0,1]$ ($E=2$, $\nu=0.25$) and $[1,3]\times[0,1]$ ($E=1$, $\nu=0.125$ — the same $\nu/E$, so both layers contract equally and the stress stays uniaxial), plane stress, joined by a linear interface spring $K=4$ (normal and tangential); left edge $u_x=0$ (and one point $u_y=0$), traction $0.1$ on the right.
Ask an LLM for a DOLFINx 0.11 script with two submeshes and an interface integral that prints the end displacement **and checks it** against the exact value. Paste it below, run it with `-np 2`, and judge it.

In [ ]:
%%fenicsx -np 2
# Paste the LLM's script here (keep the %%fenicsx line). Then run the reference cell below.

In [ ]:
%%fenicsx -np 2
import sys; sys.path.insert(0, "/tmp/fnx")
from czm import *

msh = mesh.create_rectangle(comm, [[0, 0], [3, 1]], [12, 4], mesh.CellType.quadrilateral)
ct = tag_cells(msh, lambda x: np.where(x[0] < 1.0, 1, 2))
b = Bimaterial(msh, ct, E=(2.0, 1.0), nu=(0.25, 0.125), law="linear", K=4.0)
b.fix(0, lambda x: np.isclose(x[0], 0.0), 0); b.fix_point(0, lambda x: np.isclose(x[0], 0) & np.isclose(x[1], 0), 1)
b.setup(load_side=1); r = b.solve_load(0.1)
say(f"u_end = {b.end_disp():.12f};  exact 0.1 (1/2 + 2/1) + 0.1/4 = {0.1 * (1 / 2 + 2 / 1) + 0.1 / 4:.12f};  SNES {r}")

> *your verdict on the LLM's code*